In [24]:
import os
model_to_test = 'MNIST_hgq2'
model_revision = 'Training_AdaptiveHP'
hls4ml_revision = 'VU'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = f"""
# Model Configuration


- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [2]:
import os
import keras
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
seed = 0
np.random.seed(seed)
import tensorflow as tf
tf.random.set_seed(seed)

In [5]:
from hgq.utils.sugar import Dataset

input_shape = (28, 28, 1)

# Load the data and split it between train and test sets, can also be binarized instead
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
#X_train, X_test = (X_train > 127).astype(np.float32), (X_test > 127).astype(np.float32)

# Scale images to the [0, 1] range
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)
print("X_train shape:", X_train.shape)
print(X_train.shape[0], "train samples")
print(X_test.shape[0], "test samples")

N_train = int(0.9 * len(X_train))
X_train, X_val = X_train[:N_train], X_train[N_train:]
y_train, y_val = y_train[:N_train], y_train[N_train:]

# Convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test, 10)
y_val = keras.utils.to_categorical(y_val, 10)

dataset_train = Dataset(X_train, y_train, batch_size=2048, device='gpu:0', shuffle=True)
dataset_val = Dataset(X_val, y_val, batch_size=2048, device='gpu:0')
dataset_test = Dataset(X_test, y_test, batch_size=2048, device='gpu:0')

X_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1778070874.426054   51243 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [3]:
from keras.models import Sequential
from keras.optimizers import Adam
from keras.losses import SparseCategoricalCrossentropy
from hgq.config import QuantizerConfig, QuantizerConfigScope, LayerConfigScope
from hgq.layers import QDense, QConv2D
from hgq.constraints import MinMax

In [ ]:
with (
    QuantizerConfigScope(place='all', default_q_type='kbi',overflow_mode='SAT_SYM',heterogeneous_axis=None,homogeneous_axis=(),trainable=True),
    QuantizerConfigScope(place='datalane', default_q_type='kif', overflow_mode='WRAP', f0=8, i0=4, k0=0, fc=MinMax(2, 10), ic=MinMax(2, 6), heterogeneous_axis=(), homogeneous_axis=None, trainable=True),
    LayerConfigScope(enable_ebops=True, beta0=1e-6),
   ):

    inputs = keras.Input(shape=input_shape)

    x = QConv2D(
        16, (3, 3),
        activation='relu',
        name='conv0',
        kernel_regularizer=keras.regularizers.L2(1e-3)
    )(inputs)

    x = keras.layers.MaxPooling2D((2, 2), name='pool0')(x)

    x = QConv2D(
        16, (3, 3),
        activation='relu',
        name='conv1',
        kernel_regularizer=keras.regularizers.L2(1e-3)
    )(x)

    x = keras.layers.MaxPooling2D((2, 2), name='pool1')(x)

    x = keras.layers.Flatten()(x)
    x = keras.layers.Dropout(0.2)(x)

    outputs = QDense(
        10,
        name='dense0'
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
    jit_compile=True,
    steps_per_execution=10
)

In [17]:
model.summary()

# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv0 (QConv2D)                 │ (None, 26, 26, 16)     │         2,995 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool0 (MaxPooling2D)            │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (QConv2D)                 │ (None, 11, 11, 16)     │         9,286 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense0 (QDense)                 │ (None, 10)             │        16,046 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,327 (89.34 KB)

 Trainable params: 20,256 (79.12 KB)

 Non-trainable params: 8,071 (10.21 KB)

In [25]:
import keras
import numpy as np
from math import cos, pi
from hgq.utils.sugar import BetaScheduler, Dataset, FreeEBOPs, ParetoFront, PBar, PieceWiseSchedule, EarlyStoppingWithEbopsThres
from keras.callbacks import CSVLogger, LearningRateScheduler

OUTPUT_PATH_PARETO = os.path.join(model_dir, f'model_outputs')
OUTPUT_PATH_LOG = os.path.join(model_dir, f"{str(model_to_test)}_{str(model_revision)}_log.csv")

if not os.path.exists(OUTPUT_PATH_PARETO):
    os.makedirs(OUTPUT_PATH_PARETO)
    print(f"Created new folder: {OUTPUT_PATH_PARETO}")

def cosine_decay_restarts_schedule(
    initial_lr, first_decay_steps, m_mul, alpha
):
    def schedule(step):
        cycle = step // first_decay_steps
        x = (step % first_decay_steps) / first_decay_steps
        lr = alpha + 0.5 * (initial_lr * (m_mul ** cycle) - alpha) * (1 + cos(pi * x))
        return lr
    return schedule
def warmup_cosine_schedule(
    initial_lr,
    warmup_steps,
    first_decay_steps,
    m_mul=0.9,
    alpha=1e-5
):
    cosine = cosine_decay_restarts_schedule(
        initial_lr,
        first_decay_steps,
        m_mul=m_mul,
        alpha=alpha
    )

    def schedule(step):
        # Warmup phase
        if step < warmup_steps:
            return initial_lr * (0.1 + 0.9 * step / warmup_steps)

        # Cosine phase
        return cosine(step - warmup_steps)

    return schedule

pbar = PBar(
        'loss: {loss:.2f}/{val_loss:.2f} - acc: {accuracy:.4f}/{val_accuracy:.4f} - lr: {learning_rate:.2e} - beta: {beta:.1e}'
    )
ebops = FreeEBOPs()
pareto = ParetoFront(
        OUTPUT_PATH_PARETO,
        ['val_accuracy', 'ebops'],
        [1, -1],
        enable_if=lambda logs: logs.get("val_accuracy", 0) > 0.90,
        fname_format='epoch={epoch}-val_acc={val_accuracy:.3f}-ebops={ebops}.keras',
    )
estop = EarlyStoppingWithEbopsThres(ebops_threshold=3000, monitor='val_accuracy', patience=30, restore_best_weights=True)
beta_sched = BetaScheduler(
    PieceWiseSchedule([
        (0,    1e-7, 'constant'),
        (50,   1e-7, 'log'),
        (1000,  1e-6, 'log'),
        (5000, 1e-5, 'log'),
        (8000, 1e-4, 'constant'),
    ])
)
lr_sched = LearningRateScheduler(
    warmup_cosine_schedule(
        initial_lr=1e-3,
        warmup_steps=50,
        first_decay_steps=250,
        m_mul=0.9,
        alpha=5e-4
    )
)
csv_logger = CSVLogger(OUTPUT_PATH_LOG, append=True, separator=';')

#Fixed HP training without beta_sched and lr_sched.
#Adaptive HP training utilizes schedulers for beta and learning rate with cosine decay restarts.
callbacks = [ebops, lr_sched, beta_sched, pbar, pareto, estop, csv_logger]

In [7]:
model.fit(dataset_train, epochs=8000, validation_data=dataset_val, callbacks=callbacks, verbose=0)

  0%|          | 0/8000 [00:00<?, ?epoch/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1777928612.296762 1167655 service.cc:148] XLA service 0x7fc8841770c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777928612.296843 1167655 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-05-04 23:03:32.411797: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777928612.911258 1167655 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-04 23:03:34.046745: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1951', 48 bytes spill stores, 48 bytes spill loads

2026-05-04 23:03:34.334830: I external/local_xla/xla/stre

In [30]:
import os, glob
from keras.models import load_model
from hgq.utils import trace_minmax

# 1. Get all Pareto models
files = glob.glob(os.path.join(OUTPUT_PATH_PARETO,'*.keras'))

# 2. Sort them by the EBOPs number in the filename
# This assumes filename: "...-ebops=123.45-..."
files.sort(key=lambda x: float(x.split('ebops=')[1].split('-')[0]))
smallest = files[0]
print(f"Done! Picked {smallest}")

model_export = load_model(smallest)
ebops = trace_minmax(model_export, X_test, verbose=False, return_results=False)

score = model_export.evaluate(dataset_test, verbose=0)
print("Total EBOPs:", ebops)
print("Test loss:", score[0])
print("Test accuracy:", score[1])
test_acc = score[1]

model_name = f"model_{model_revision}_acc={test_acc:.4f}_ebops={ebops:.0f}.keras"
model_export.save(os.path.join(model_dir, model_name))

Done! Picked /home/slopin/HLS4ML_testbench_KV260/development/MNIST_CNN/MNIST_hgq2/Training_AdaptiveHP/model_outputs/epoch=3273-val_acc=0.961-ebops=30908-val_loss=0.136.keras
Total EBOPs: 34304
Test loss: 0.14395134150981903
Test accuracy: 0.9577999711036682
